In [4]:
import ConnectionConfig as cc
from pyspark.sql.functions import *
cc.setupEnvironment()
cc.listEnvironment()

HOMEBREW_PREFIX: /opt/homebrew
COMMAND_MODE: unix2003
INFOPATH: /opt/homebrew/share/info:
SHELL: /bin/zsh
PYTHONPATH: /Users/alexandra.miron/PycharmProjects/data4team21:/Applications/PyCharm.app/Contents/plugins/python-ce/helpers/pydev:/Applications/PyCharm.app/Contents/plugins/python/helpers-pro/jupyter_debug
__CFBundleIdentifier: com.jetbrains.pycharm
TMPDIR: /var/folders/nh/bnybds3d02s2103s2s3xkdyw0000gp/T/
LC_ALL: en_US.UTF-8
HOME: /Users/alexandra.miron
HOMEBREW_REPOSITORY: /opt/homebrew
PATH: /Users/alexandra.miron/PyCharmMiscProject/.venv/bin:/opt/homebrew/bin:/opt/homebrew/sbin:/Users/alexandra.miron/tools/spark-3.5.2-bin-hadoop3/bin:/Users/alexandra.miron/tools/hadoop-3.4.0-win10-x64/bin:/Library/Frameworks/Python.framework/Versions/3.11/bin:/usr/local/bin:/System/Cryptexes/App/usr/bin:/usr/bin:/bin:/usr/sbin:/sbin:/var/run/com.apple.security.cryptexd/codex.system/bootstrap/usr/local/bin:/var/run/com.apple.security.cryptexd/codex.system/bootstrap/usr/bin:/var/run/com.apple.sec

In [5]:
spark = cc.startLocalCluster("dimWeather")
spark.getActiveSession()

In [7]:
# EXTRACT
df_weather_dim = spark.read.option("header", True).csv("weather_dim.csv")

# TRANSFORM
# surrogate key (Spark doesn't have SERIAL, so we mimic it)
from pyspark.sql.functions import monotonically_increasing_id

df_weather_dim = df_weather_dim.withColumn("weather_id", monotonically_increasing_id())

# Inspect schema and preview
df_weather_dim.printSchema()
df_weather_dim.show()

# LOAD
df_weather_dim.write.format("delta").mode("overwrite").saveAsTable("WeatherDim")


root
 |-- weather_type: string (nullable = true)
 |-- weather_id: long (nullable = false)

+------------+----------+
|weather_type|weather_id|
+------------+----------+
|    Pleasant|         0|
|  Unpleasant|         1|
|     Neutral|         2|
|     Unknown|         3|
+------------+----------+



In [8]:
print(cc.create_jdbc())

jdbc:postgresql://localhost:5432/tutorial_op?user=postgres&password=admin1234&ssl=false


## Delete the spark session

In [9]:
spark.stop()